# Shoplifting Detection: Video Captioning using Ollama & Qwen3.5 2B MLX

This notebook demonstrates how to use the local Ollama vision-language model (`qwen3.5:2b-mlx`) to analyze and caption a surveillance video. The goal is to identify suspicious behaviors (like shoplifting, concealing items, or nervous movements) in a sample video.

### Methodology
1. **Frame Extraction**: Extract frames from the video file at a regular temporal interval (e.g., every 3 seconds) to avoid redundant frame processing and reduce computational load.
2. **Temporary Storage**: Save these frames to temporary JPEG files on disk. The Ollama Python SDK reads images directly from file paths, which is the most reliable way to transmit multi-modal data to the local server.
3. **Multi-modal Chat Inference**: Call the local `qwen3.5:2b-mlx` model using Ollama's Python library, providing the prompt and the absolute path to each frame image.
4. **Analysis & Cleanup**: Display the timestamp and model description for each frame, and immediately delete the temporary image files.

In [ ]:
import os
import cv2
import ollama

# Define paths and configuration
video_path = "sample_videos/gettyimages-1995820194-640_adpp.mp4"
model_name = "qwen3.5:2b-mlx"
interval_seconds = 3.0  # Extract one frame every 3 seconds for analysis

print(f"Using video: {video_path}")
print(f"Using Ollama vision model: {model_name}")

In [ ]:
# Inspect video properties
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video file {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = total_frames / fps
print(f"Video Properties:")
print(f"  - FPS: {fps:.2f}")
print(f"  - Total Frames: {total_frames}")
print(f"  - Duration: {duration:.2f} seconds")
cap.release()

In [ ]:
def analyze_video_frames(video_path, model_name, interval_seconds):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    step = int(fps * interval_seconds)
    if step < 1:
        step = 1

    prompt = (
        "Analyze this frame from a surveillance camera. "
        "Describe what the person is doing in 1-2 concise sentences. "
        "Pay close attention to suspicious activities such as shoplifting, "
        "taking items from shelves, concealing items, or looking around nervously."
    )

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % step == 0:
            timestamp = frame_idx / fps
            temp_filename = f"temp_frame_{timestamp:.2f}s.jpg"
            temp_path = os.path.abspath(temp_filename)
            
            # Save frame to disk
            cv2.imwrite(temp_path, frame)
            print("-" * 60)
            print(f"Analyzing frame at timestamp {timestamp:.2f}s...")
            
            try:
                # Query Ollama vision model using the absolute file path
                response = ollama.chat(
                    model=model_name,
                    messages=[
                        {
                            'role': 'user',
                            'content': prompt,
                            'images': [temp_path]
                        }
                    ]
                )
                caption = response['message']['content'].strip()
                print(f"Model Caption:\n{caption}")
            except Exception as e:
                print(f"Error calling Ollama: {e}")
            finally:
                # Clean up temporary frame image
                if os.path.exists(temp_path):
                    os.remove(temp_path)
                    print(f"[Info] Cleaned up temporary file: {temp_filename}")
        
        frame_idx += 1
    cap.release()

# Run the analysis
analyze_video_frames(video_path, model_name, interval_seconds)